# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadhany222/flyrank-ml-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content item, on one day** (grain: `report_date × client_hash_id × content_hash_id`
in `fact_content_daily_performance`). I'm working over **month=2026-03** — a mid-panel month, not
the final month (`_sample`), so I'm not accidentally developing inside my own future outcome window.
I'll verify both the grain and the date span with real queries in Section 3.

## 2. Fields: feature / label / context / excluded

**Table(s):** `fact_content_daily_performance` (month=2026-03 partition) for daily signals,
joined to `dim_content` for content-level context when needed.

**Target/proxy:** whether a content item's impressions declined within the month — a proxy,
not a true future outcome (same honesty flag as w01/w02: this is a within-window label, not
a forward-predicting one).

**One deliberate exclusion:** I'm excluding `gsc_avg_position` rows where the value represents
"no data" rather than a real rank — mixing those in would corrupt any position-based feature.
I'm also excluding any FlyRank product decision flags (`health_score`, `priority_score`, etc.) —
per the lane guide, those aren't even shipped in this data, but I'm naming the exclusion
explicitly so it's clear I know why.

**Field buckets:**
- **Feature:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engagement_rate` — all knowable during the observed window, before any "outcome" is decided.
- **Label/proxy:** within-month impressions trend (declining vs not) — computed from the same signals, never fed back in as a feature.
- **Context:** `client_hash_id`, `content_hash_id`, `report_date` — for joining/grouping/splitting only, never model inputs.
- **Excluded:** any product-decision fields (none shipped here, per the lane guide) and `gsc_avg_position` rows with no real data.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Three checks below prove the contract above is true, not assumed. Then five features, each with
an "available at decision time" line. Then the leakage trap, performed and undone.

In [20]:

import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected. Using month=2026-03 (mid-panel, not the sealed final month).")


Connected. Using month=2026-03 (mid-panel, not the sealed final month).


In [21]:

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MONTH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Duplicate grain rows found: {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,c


In [22]:
span = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {FACT_MONTH}
""").df()

print(span.to_string(index=False))

 row_count   min_date   max_date
   9841378 2026-03-01 2026-03-31


In [23]:

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {FACT_MONTH}
""").df()

print(availability.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  ga4_available_rows  pct_available
    9841378            413966.0            4.2


### Five features (max), each knowable at decision time

1. **`imp_month`** (sum of `gsc_impressions` over the month) — available because it's a direct
   observed count from Search Console, known as soon as the month's data lands.
2. **`clicks_month`** (sum of `gsc_clicks`) — same: an observed count, no future information needed.
3. **`avg_position_month`** (average of `gsc_avg_position`, excluding no-data rows) — available
   because position is measured continuously during the window, not derived from an outcome.
4. **`sessions_month`** (sum of `ga4_sessions`, only where `ga4_data_available IS TRUE`) —
   available for clients with active GA4 tracking during this window; explicitly filtered so
   zero-fill rows don't get treated as real zero engagement.
5. **`ctr_month`** (`clicks_month / imp_month`) — a derived ratio from two features already
   available at decision time, not from anything in a future window.

In [24]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_month,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE NULL END) AS sessions_month
    FROM {FACT_MONTH}
    GROUP BY 1, 2
    HAVING imp_month >= 50
""").df()

features['ctr_month'] = features['clicks_month'] / features['imp_month']

print(f"{len(features):,} content items with enough monthly volume")
features.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,114 content items with enough monthly volume


,client_hash_id,content_hash_id,imp_month,clicks_month,avg_position_month,sessions_month,ctr_month
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,NaN,0.001754
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,7.842593,NaN,0.000000
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,8.454069,4.0,0.000000
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,0.004222
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.005776
5,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,7.046534,1.0,0.006667
6,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,0.0,19.563725,NaN,0.000000
7,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,4.950311,8.0,0.003803
8,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,52.127896,1.0,0.000000
9,client_73cda7b4e4f265ea,content_3dba50ae010f3f30,357.0,1.0,20.926174,NaN,0.002801


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

trend = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {FACT_MONTH}
    GROUP BY 1, 2
""").df()

trend['is_declining'] = (trend['imp_second_half'] < 0.8 * trend['imp_first_half']).astype(int)

data = features.merge(trend, on=['client_hash_id', 'content_hash_id'])

honest_cols = ['imp_month', 'avg_position_month', 'sessions_month']
clean = data.dropna(subset=honest_cols + ['is_declining'])

X, y = clean[honest_cols], clean['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"HONEST ROC-AUC (no leak): {honest_auc:.3f}")

leaky_cols = honest_cols + ['imp_second_half']
clean_leaky = data.dropna(subset=leaky_cols + ['is_declining'])
print(f"Rows available for leaky model: {len(clean_leaky)}")

Xl, yl = clean_leaky[leaky_cols], clean_leaky['is_declining']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.25, random_state=42, stratify=yl)
leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xl_tr, yl_tr)
leaky_auc = roc_auc_score(yl_te, leaky_model.predict_proba(Xl_te)[:, 1])
print(f"LEAKY ROC-AUC (imp_second_half included): {leaky_auc:.3f}")

print(f"\nFinal kept result: HONEST ROC-AUC = {honest_auc:.3f} (imp_second_half removed from features)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST ROC-AUC (no leak): 0.595
Rows available for leaky model: 59850
LEAKY ROC-AUC (imp_second_half included): 0.998

Final kept result: HONEST ROC-AUC = 0.595 (imp_second_half removed from features)


## 4. Data limits
**Named limitation:** history depth differs wildly by client (an unbalanced panel) — some
clients have 12+ months of data, others much less, and rows before a client's `ga4_data_start`
are zero-filled with `ga4_data_available = FALSE` rather than truly having zero engagement.
This means comparing engagement-based features across clients without checking their tracking
start dates would silently mix "no engagement" with "no tracking yet" — a real trap this
month-level slice can't protect you from unless you filter on the availability flag every time,
which I did above in the feature query.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.